1. Charger taxis, convertir pickup et dropoff en dates

In [1]:
import pandas as pd
import seaborn as sns
import ssl

ssl._create_default_https_context = ssl._create_unverified_context
taxis = sns.load_dataset("taxis")

taxis['pickup'] = pd.to_datetime(taxis['pickup'])
taxis['dropoff'] = pd.to_datetime(taxis['dropoff'])
print(taxis.dtypes)

pickup             datetime64[us]
dropoff            datetime64[us]
passengers                  int64
distance                  float64
fare                      float64
tip                       float64
tolls                     float64
total                     float64
color                         str
payment                       str
pickup_zone                   str
dropoff_zone                  str
pickup_borough                str
dropoff_borough               str
dtype: object


2. Calculer le nombre de courses et le chiffre d'affaires par jour

In [2]:
# On garde seulement la date (sans l'heure)
taxis['jour'] = taxis['pickup'].dt.date

courses_par_jour = taxis.groupby('jour')['total'].count()
print(courses_par_jour)

chiffre_affaires_par_jour = taxis.groupby('jour')['total'].sum()
print(chiffre_affaires_par_jour)

jour
2019-02-28      1
2019-03-01    241
2019-03-02    198
2019-03-03    169
2019-03-04    171
2019-03-05    228
2019-03-06    257
2019-03-07    218
2019-03-08    235
2019-03-09    204
2019-03-10    185
2019-03-11    209
2019-03-12    218
2019-03-13    244
2019-03-14    260
2019-03-15    201
2019-03-16    220
2019-03-17    178
2019-03-18    173
2019-03-19    201
2019-03-20    233
2019-03-21    221
2019-03-22    229
2019-03-23    209
2019-03-24    149
2019-03-25    155
2019-03-26    178
2019-03-27    232
2019-03-28    205
2019-03-29    209
2019-03-30    215
2019-03-31    187
Name: total, dtype: int64
jour
2019-02-28       6.30
2019-03-01    4213.83
2019-03-02    3319.02
2019-03-03    3027.32
2019-03-04    3269.08
2019-03-05    4377.38
2019-03-06    4673.61
2019-03-07    4115.70
2019-03-08    4524.21
2019-03-09    3785.80
2019-03-10    3198.69
2019-03-11    4021.26
2019-03-12    3875.80
2019-03-13    4868.06
2019-03-14    4742.55
2019-03-15    4030.29
2019-03-16    3836.58
2019-03-17    

3. Trouver l'heure de la journée la plus chargée

In [3]:
taxis['heure'] = taxis['pickup'].dt.hour

# idxmax() donne l'heure qui a le plus de courses
heure_plus_chargee = taxis['heure'].value_counts().idxmax()
print(f"Heure la plus chargée : {heure_plus_chargee}h")

Heure la plus chargée : 18h


4. Quel jour de la semaine a le pourboire moyen le plus élevé ? 

In [4]:
taxis['jour_semaine'] = taxis['pickup'].dt.day_name()

pourboire_moyen_par_jour = taxis.groupby('jour_semaine')['tip'].mean()
print(pourboire_moyen_par_jour)

meilleur_jour_pourboire = pourboire_moyen_par_jour.idxmax()
print(f"Jour avec le pourboire moyen le plus élevé : {meilleur_jour_pourboire}")

jour_semaine
Friday       1.980108
Monday       2.027316
Saturday     1.815143
Sunday       1.908710
Thursday     2.062177
Tuesday      2.001370
Wednesday    2.087329
Name: tip, dtype: float64
Jour avec le pourboire moyen le plus élevé : Wednesday


5. Calculer la durée de chaque course en minutes, repérer et traiter les durées aberrantes

In [5]:
taxis['duree_minutes'] = (taxis['dropoff'] - taxis['pickup']).dt.total_seconds() / 60
print(taxis['duree_minutes'].describe())

# Une course de 0 minute est impossible : on la supprime
# Le maximum (environ 108 minutes) reste possible : on garde le reste
courses_aberrantes = (taxis['duree_minutes'] <= 0).sum()
print(f"Courses de 0 minute : {courses_aberrantes}")

taxis = taxis[taxis['duree_minutes'] > 0]
print(f"Courses restantes : {len(taxis)}")

count    6433.000000
mean       14.349617
std        11.643892
min         0.000000
25%         6.500000
50%        10.900000
75%        18.516667
max       107.666667
Name: duree_minutes, dtype: float64
Courses de 0 minute : 6
Courses restantes : 6427
